


<div class="alert alert-info" role="alert">

 ## Travail à réaliser:

***Questions préliminaires (4 points)***

Dans le Branch-and-Bound fourni dans le notebook-exemple (⚠️ pas dans le code que vous avez modifié) :

- Quelle est la règle de séparation choisie?  
- Quelle méthode de calcul de borne supérieure est utilisée?  
- Quels sont les tests de sondabilité TA, TO, TR?  
- Quelle est la stratégie d'exploration choisie



***Rédaction (5 points)***

Rédigez une réflexion critique :
- Quels points du notebook pourraient être améliorés ?
- Faites le lien avec les notions vues lors du TD 2.
- Quelles pistes d’optimisation ou d’extension envisagez-vous ?



***Code – Amélioration (6 points)***

- Modifiez le notebook initial pour y apporter vos propres améliorations.
- Pensez à commenter votre code.


***Code – Comparaison (5 points)***

Comparez les performances du Branch-and-Bound entre :
- la version initiale du notebook,
- votre version améliorée.

Pour cela :
- utilisez un nombre suffisant d’instances et de tailles variées,
- discutez les résultats obtenus,
- indiquez si ces résultats vous semblent cohérents et justifiez vos arguments.



***Rendu attendu***

Un notebook unique contenant :
- vos réponses aux questions (sous forme de cellules Markdown),
- la rédaction,
- le code initial,
- vos améliorations avec commentaires,
- la comparaison des performances.
</div>

# TP2 — Branch-and-Bound (B&B) pour le sac à dos
<div class="alert-success">
<b>😀 Objectif :</b> Comprendre et implémenter un Branch-and-Bound complet pour le sac à dos.
</div>

## 🌳 Pourquoi le Branch-and-Bound ? 
- <b>But</b> : éviter l'énumération exhaustive (2^n feuilles, n = nb objets) en identifiant tôt les branches inutiles.   

<div class="alert-success">
<b>🧐 À retenir :</b> une <i>bonne borne</i> + une <i>bonne stratégie</i> = un arbre <i>beaucoup</i> plus petit.
</div>


Exemple du TD:

4 objets et la capacité est 10 tel que: 
- Objet 1 - valeur: 42, poids: 7
- Objet 2 - valeur: 40, poids: 4
- Objet 3 - valeur:12, poids: 3
- Objet 4 - valeur:25, poids: 5

Énumération exhaustive des $2^4 = 16$ combinaisons

| Combinaison {1,2,3,4} | Objets Inclus  | Poids Total      | Valeur Totale       | Admissible (Poids ≤ 10) ? |
| :------------------: | :-------------: | :--------------: | :-----------------: | :-----------------------: |
|        `0000`        |       {}        |        0         |          0          |          ✅ Oui           |
|        `0001`        |       {4}       |        5         |         25          |          ✅ Oui           |
|        `0010`        |       {3}       |        3         |         12          |          ✅ Oui           |
|        `0011`        |     {3, 4}      |    3 + 5 = 8     |     12 + 25 = 37    |          ✅ Oui           |
|        `0100`        |       {2}       |        4         |         40          |          ✅ Oui           |
|      **`0101`** |   **{2, 4}** |  **4 + 5 = 9** |  **40 + 25 = 65** |        **✅ Oui** |
|        `0110`        |     {2, 3}      |    4 + 3 = 7     |     40 + 12 = 52    |          ✅ Oui           |
|        `0111`        |    {2, 3, 4}    |   4 + 3 + 5 = 12   |    40+12+25 = 77    |          ❌ Non           |
|        `1000`        |       {1}       |        7         |         42          |          ✅ Oui           |
|        `1001`        |     {1, 4}      |    7 + 5 = 12    |     42 + 25 = 67    |          ❌ Non           |
|        `1010`        |     {1, 3}      |    7 + 3 = 10    |     42 + 12 = 54    |          ✅ Oui           |
|        `1011`        |    {1, 3, 4}    |   7 + 3 + 5 = 15   |    42+12+25 = 79    |          ❌ Non           |
|        `1100`        |     {1, 2}      |    7 + 4 = 11    |     42 + 40 = 82    |          ❌ Non           |
|        `1101`        |    {1, 2, 4}    |   7 + 4 + 5 = 16   |   42+40+25 = 107    |          ❌ Non           |
|        `1110`        |    {1, 2, 3}    |   7 + 4 + 3 = 14   |    42+40+12 = 94    |          ❌ Non           |
|        `1111`        |   {1, 2, 3, 4}  | 7 + 4 + 3 + 5 = 19 | 42+40+12+25 = 119 |          ❌ Non           |

## 🔗 Lien avec la programmation linéaire
- La relaxation linéaire d’un PLNE fournit une <b>borne supérieure</b> (en maximisation) 



## Stratégie Branch and Bound
<div class="alert-danger">
🧐 Important :
    
* **Calcul de Borne :** De quelle manière relâche-t-on le problème pour obtenir une borne ? 
* **Règle de Séparation :** Comment diviser le problème  ?
* **Règle d’Exploration :** Quel est le prochain nœud à explorer ?
* **Tests de Sondabilité :** Faut-il abandonner cette branche ?
  
</div>

## Dans ce notebook
I - [Initialisation et récupération des données](#Initialisation)   
II - [Tests de sondabilités TA, TO et TR basés sur le modèle linéaire](#Tests)  
III - [Règle Exploration et Séparation](#SeparationEtExploration)  
IV - [Calcul de la borne](#Borne)  
V - [Creation de l'arbre et résolution](#Arbre)  
VI - [Affichage du résultat final](#Affichage)

## I - Initialisation et récupération des données  <a name="Initialisation"></a>

### Initialisation (à faire une seule fois)

In [1]:
import Pkg; 
Pkg.add("GraphRecipes"); Pkg.add("Plots"); 
using GraphRecipes, Plots #only used to visualize the search tree at the end of the branch-and-bound

LoadError: IOError: open("/mnt/n7fs/ens/tp_cots/packages/julia-1.11.4-2025/logs/manifest_usage.toml.pid", 194, 292): permission denied (EACCES)

### Récupération des données

In [2]:
# Reads knapsack .txt instances.
function readKnaptxtInstance(filename)
    price=Int64[]
    weight=Int64[]
    KnapCap=Int64[]
    open(filename) do f
        # Iterate over the instance file 3 lines: prices, weights, capacity.
        for i in 1:3
            tok = split(readline(f))
            if(tok[1] == "ListPrices=")
                for i in 2:(length(tok)-1)
                    push!(price,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "ListWeights=")
                for i in 2:(length(tok)-1)
                    push!(weight,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "Capacity=")
                push!(KnapCap, parse(Int64, tok[2]))
            else
                println("Unknown read :", tok)
            end
        end
    end
    capacity=KnapCap[1]
    
    return price, weight, capacity
end

readKnaptxtInstance (generic function with 1 method)

## II -  Tests de sondabilités TA, TO et TR <a name="Tests"></a>

In [3]:
# Sondability test.
function testSondability(BestProfit, Bestsol, capacity, bornesup, listVals, listObjs, price)
    TA, TO, TR = false, false, false
    if(capacity < 0)
        TA=true
        println("TA")
    elseif( bornesup <= BestProfit)
        TO=true
        println("TO")
    elseif all( v == 0.0 || v == 1.0 for v in listVals)
        TR=true
        println("TR")
        if (bornesup > BestProfit)
            sol = zeros(length(price))
            for i=1 :length(listVals)
                    sol[listObjs[i]] = listVals[i]
            end
            Bestsol = sol
            BestProfit = 0
            for i = 1:min(length(listVals),length(price))
                BestProfit += listVals[i]*price[listObjs[i]]
            end
            println("\nNew Solution memorized ", Bestsol, " with bestprofit ", BestProfit, "\n")
        else 
            println("test")
        end

    else
        println("non sondable")
    end
    TA, TO, TR, Bestsol, BestProfit
end

testSondability (generic function with 1 method)

## III - Procédure de séparation (branching) et stratégie d'exploration permettant de se placer au prochain noeud à traiter  <a name="SeparationEtExploration"></a>

In [4]:
# Branching method.
function separateNodeThenchooseNext_lexicographic_depthfirst!(listObjs, listVals, n)
    i, obj = 1, 0
    while((i <= n) && (obj==0))
        if(!(i in listObjs))
            obj=i
        end
        i+=1
    end
    println("\nbranch on object ", obj, "\n")
    push!(listObjs,obj) 
    push!(listVals,1.0) 
end

# Exploration method.
function exploreNextNode_depthfirst!(listObjs, listVals, listNodes)
    stop=false
    if (length(listObjs)>= 1)
        obj=pop!(listObjs)
        theval=pop!(listVals)
        tmp=pop!(listNodes)
        while( (theval==0.0) && (length(listObjs)>= 1))
            obj=pop!(listObjs)
            theval=pop!(listVals)
            tmp=pop!(listNodes)
        end
        if theval==1.0
            push!(listObjs,obj)
            push!(listVals,0.0)
        else
            println("\nFINISHED")
            stop=true
        end
    else
        println("\nFINISHED")
        stop=true
    end
    return stop
end

exploreNextNode_depthfirst! (generic function with 1 method)

###  IV - Calcul de la borne  <a name="Borne"></a>

In [5]:
# List knapsack objects by their best price/weight ratio.
function list_best_ratio(price, weight)
    n = length(price)
    listRatio = zeros(n)
    for i in 1:n
        listRatio[i] = (price[i]/weight[i])
    end
    return sortperm(listRatio, rev = true)
end

# Upper bound computation.
function bound_1(price,weight,capacity, listObjs, listVals)
    bestRatio = list_best_ratio(price, weight)
    for obj in bestRatio
        if !(obj in listObjs)
            amountTaken = capacity / weight[obj]
            push!(listObjs, obj)
            push!(listVals, amountTaken)
            break
        end
    end
   
    upperBound = 0 
    for i=1:(min(length(price),length(listObjs)))
        upperBound += price[listObjs[i]]*listVals[i]
    end

    return upperBound, listObjs, listVals
end

# Display the model being solved.
function display_model(price, weight, capacity, listObjs, listVals)
    print("Max ")
    n = length(price)
    for i=1:n
        if (!(i in listObjs))
            print(price[i] , " x[" , i , "]")
            if i < n
                print(" + ")
            end
        end
    end

    println()
    println("Subject To")
    for i=1:n
        if (!(i in listObjs))
            print(weight[i] , " x[" , i , "]")
            if i < n
                print(" + ")
            end
        end
    end
    print(" <= " , capacity)
    println()
end

display_model (generic function with 1 method)

### V - Création de l'arbre et résolution  <a name="Arbre"></a>


Boucle principale : résoudre une relaxation, appliquer les tests de sondabilité, identifier le prochain noeud, répéter.

In [6]:
# Knapsack solve.
function solveKnapInstance(filename)

    price, weight, capacity = readKnaptxtInstance(filename)

    # Vizualization tools to memorize the search tree to display (optional).
    trParentnodes=Int64[] # Stores orig node of arcs in the search tree.
    trChildnodes=Int64[] # Stores destination node of arcs in the search tree.
    trNamenodes=[] # Stores names of nodes in the search tree.

    # Intermediate structures.
    listObjs=Int64[] # Objects selected at each node.
    listVals=Float64[] # Knapsack value at each node.
    listNodes=Int64[] # Nodes number.

    BestProfit::Float64=-1.0
    Bestsol=Float64[]

    capacity0 = capacity
    current_node_number::Int64=0
    stop = false

    while(!stop)

        println("\nNode number ", current_node_number, ": \n---------------\n")

        # Update the graphical tree.
        push!(trNamenodes,current_node_number+1) 
        if(length(trNamenodes)>=2)
            push!(trParentnodes,listNodes[end]+1)
            push!(trChildnodes, current_node_number+1)
        end
        push!(listNodes, current_node_number)

       
        # Calcul de la capacité pour chaque noeud 
        capacity = capacity0
        for i=1 :length(listVals)
            capacity = capacity- listVals[i]*weight[listObjs[i]]
        end

        display_model(price, weight, capacity, listObjs, listVals)
        
        n = length(price)
        m = length(listVals)
        full = (n == m)
        
        # Upper bound computation.
        bornesup , listObjs , listVals = bound_1(price,weight,capacity, listObjs,listVals)

        println("\nPrevious Solution memorized ", Bestsol, " with bestprofit ", BestProfit, "\n")
       
        # Sondability tests.
        TA, TO, TR, Bestsol, BestProfit = testSondability(BestProfit, Bestsol, capacity, bornesup, listVals, listObjs, price)
        is_node_sondable = TA || TO || TR
        
        # Pop last value and objects added by the upper bound computation.
        if !full
            pop!(listVals)
            pop!(listObjs)
        end
        
        # Branching and exploration.
        if(!is_node_sondable)
            separateNodeThenchooseNext_lexicographic_depthfirst!(listObjs, listVals, length(price))
        else
            stop = exploreNextNode_depthfirst!(listObjs, listVals, listNodes)
        end
        
        current_node_number = current_node_number + 1
    end

    println("\n******\n\nOptimal value = ", BestProfit, "\n\nOptimal x=", Bestsol)
    println("Nombre de noeuds : ", current_node_number)
    Indices = Int64[]
    for i=1 : length(Bestsol)
        if Bestsol[i] == 1.0
            push!(Indices,i)
        end
    end
    println("Objets selectionnés : ", Indices)
    println("Meilleur profit : ", BestProfit)
    return BestProfit, Bestsol, trParentnodes, trChildnodes, trNamenodes

end


solveKnapInstance (generic function with 1 method)

### VI - Affichage du résultat final <a name="Affichage"></a>

In [7]:
# Solve and display of knapsack.
function solveNdisplayKnap(filename)

    println("\n Branch-and-Bound for solving a knapsack problem. \n\n Solving instance '" * filename * "'\n")

    BestProfit, Bestsol, trParentnodes, trChildnodes, trNamenodes = solveKnapInstance(filename)

    println("\n******\n\nOptimal value = ", BestProfit, "\n\nOptimal x=", Bestsol)

    println("\n Branch-and-bound tree visualization : start display ...")
    display(graphplot(trParentnodes, trChildnodes, names=trNamenodes, method=:tree))
    println("... end display. \n\n")

end

solveNdisplayKnap (generic function with 1 method)

In [8]:
# Main.
instance = "InstancesKnapSack/test.opb.txt"
solveNdisplayKnap(instance)
println("press enter to exit ! ")
readline()


 Branch-and-Bound for solving a knapsack problem. 

 Solving instance 'InstancesKnapSack/test.opb.txt'


Node number 0: 
---------------

Max 42 x[1] + 40 x[2] + 12 x[3] + 25 x[4]
Subject To
7 x[1] + 4 x[2] + 3 x[3] + 5 x[4] <= 10

Previous Solution memorized Float64[] with bestprofit -1.0

non sondable

branch on object 1


Node number 1: 
---------------

Max 40 x[2] + 12 x[3] + 25 x[4]
Subject To
4 x[2] + 3 x[3] + 5 x[4] <= 3.0

Previous Solution memorized Float64[] with bestprofit -1.0

non sondable

branch on object 2


Node number 2: 
---------------

Max 12 x[3] + 25 x[4]
Subject To
3 x[3] + 5 x[4] <= -1.0

Previous Solution memorized Float64[] with bestprofit -1.0

TA

Node number 3: 
---------------

Max 12 x[3] + 25 x[4]
Subject To
3 x[3] + 5 x[4] <= 3.0

Previous Solution memorized Float64[] with bestprofit -1.0

non sondable

branch on object 3


Node number 4: 
---------------

Max 25 x[4]
Subject To
5 x[4] <= 0.0

Previous Solution memorized Float64[] with bestprofit -1.

LoadError: UndefVarError: `graphplot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.